In [1]:
from huggingface_hub import login
login()

In [2]:
!pip install -q --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 37.5 MB/s eta 0:00:00


In [3]:
!pip install -q \
  "llama-index>=0.11.0" \
  "llama-index-llms-huggingface" \
  "llama-index-embeddings-huggingface" \
  "llama-index-vector-stores-chroma" \
  "transformers>=4.40.0" \
  "accelerate>=0.30.0" \
  "bitsandbytes>=0.43.0" \
  "sentence-transformers" \
  "chromadb>=0.5.0" \
  "huggingface_hub"

In [4]:
!pip install -q jedi

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "microsoft/Phi-4-mini-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    load_in_4bit=True,
    torch_dtype=torch.float16
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.77G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

In [16]:
from llama_index.llms.huggingface import HuggingFaceLLM

llm = HuggingFaceLLM(
    model=model,
    tokenizer=tokenizer,
    generate_kwargs={
        "temperature": 0.7,
        "top_p": 0.9,
    },
)

In [17]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en"
)

In [18]:
from google.colab import files
uploaded = files.upload()

Saving sample_faq.txt to sample_faq.txt


In [19]:
import os

os.makedirs("data", exist_ok=True)

for file in uploaded:
    os.rename(file, f"data/{file}")

In [20]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader("data").load_data()

In [21]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(
    documents,
    embed_model=embed_model
)

In [22]:
query_engine = index.as_query_engine(
    llm=llm,
    similarity_top_k=3
)

response = query_engine.query("Explain the document in detail")
print(response)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


1. Refund Policy: The document states that the company offers a 30-day money-back guarantee on all products purchased from their store. If a customer is not completely satisfied with their purchase, they can request a full refund within 30 days of the delivery date. To request a refund, the customer needs to contact the support team at support@example.com with their order number, provide a brief reason for the refund (optional but helpful), and wait for 1-2 business days for the review. Once approved, refunds are processed within 5-7 business days and are credited back to the original payment method. Sale items and gift cards are not eligible for refunds.

2. Shipping Information: The company offers multiple shipping options to cater to different needs. Free standard shipping is available for all orders over $50, with a delivery time of 3-5 business days. Express shipping, available for an additional $15 fee, has a faster delivery time of 1-2 business days. International shipping is av

In [23]:
from llama_index.core import StorageContext

index.storage_context.persist(persist_dir="./storage")

In [25]:
from llama_index.core import load_index_from_storage, StorageContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# recreate embedding model
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en"
)

# load storage
storage_context = StorageContext.from_defaults(
    persist_dir="./storage"
)

# reload index WITH embedding model
index = load_index_from_storage(
    storage_context,
    embed_model=embed_model
)

In [28]:
!pip install -q \
  opentelemetry-api==1.38.0 \
  opentelemetry-sdk==1.38.0 \
  opentelemetry-exporter-otlp-proto-grpc==1.38.0

In [29]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

client = chromadb.Client()
collection = client.create_collection("docs")

vector_store = ChromaVectorStore(chroma_collection=collection)

storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    embed_model=embed_model
)

In [30]:
!ls -R /content

/content:
data  sample_data  storage

/content/data:
sample_faq.txt

/content/sample_data:
anscombe.json		      mnist_test.csv
california_housing_test.csv   mnist_train_small.csv
california_housing_train.csv  README.md

/content/storage:
default__vector_store.json  graph_store.json	      index_store.json
docstore.json		    image__vector_store.json


In [31]:
import json

with open("/content/storage/default__vector_store.json") as f:
    vector_data = json.load(f)

In [32]:
print(vector_data.keys())

dict_keys(['embedding_dict', 'text_id_to_ref_doc_id', 'metadata_dict'])


In [33]:
embeddings = vector_data["embedding_dict"]

print("Total embeddings:", len(embeddings))

Total embeddings: 1


In [34]:
first_id = list(embeddings.keys())[0]

print("Node ID:", first_id)
print("Embedding (first 10 values):", embeddings[first_id][:10])

Node ID: e82d6d28-dd1b-49e5-81bc-11995f2e23b7
Embedding (first 10 values): [-0.03313519060611725, -0.01653233915567398, 0.02418733574450016, 0.012929464690387249, 0.015627894550561905, -0.0050372593104839325, 0.02927570976316929, 0.04226461797952652, -0.035531315952539444, -0.02521280013024807]


In [36]:
node = docstore["docstore/data"][first_id]

print(node["__data__"]["text"])

CUSTOMER SUPPORT FAQ

=== REFUND POLICY ===

We offer a 30-day money-back guarantee on all products purchased from our store.
If you are not completely satisfied with your purchase, you can request a full refund
within 30 days of the delivery date.

To request a refund:
1. Contact our support team at support@example.com with your order number
2. Provide a brief reason for the refund (optional but helpful)
3. Our team will review your request within 1-2 business days
4. Once approved, refunds are processed within 5-7 business days
5. The refund will be credited to your original payment method

Please note: Sale items and gift cards are not eligible for refunds.


=== SHIPPING INFORMATION ===

We offer multiple shipping options to meet your needs:

FREE STANDARD SHIPPING:
- Available on all orders over $50
- Delivery time: 3-5 business days
- Tracking number provided

EXPRESS SHIPPING:
- Available for an additional $15 fee
- Delivery time: 1-2 business days
- Tracking number provided
- O